## 第 6 周 —“价格合适”顶峰 (mmaitsimwale)

### 第 6 周涵盖的内容
- **第 1 天：** 数据管理 — Amazon Reviews 2023 数据集被刮入“Item”对象（标题、类别、价格、重量、摘要）。
- **第 2 天：** 数据预处理 — 法学硕士将杂乱的产品描述重写为干净的“item.summary”文本。
- **第 3 天：** 评估工具（`pricer.evaluator.Tester`）+ 基线（随机、平均、线性回归、随机森林）。
- **第 4 天：** Frontier LLM 作为零样本定价者 - GPT-4.1-nano、Claude Opus、Gemini 2.5 Flash、Grok、GPT-5。
- **第 5 天：** 微调 — 将 JSONL 上传到 OpenAI，训练私有 GPT-4.1-nano 变体，进行评估。

### 练习任务：多提供商 LLM 价格估算基准

顶点问题：*语言模型根据产品描述估算产品价格的准确度如何？*

该笔记本电脑对**两个提供商类别**进行了基准测试：
- **OpenRouter**（付费型号）— 通往 OpenAI GPT 变体的路由
- **Groq**（免费 OSS 模型）— 运行 Llama 3.3、Llama 3.1、DeepSeek R1 Distill

每个模型都会收到相同的零样本提示。结果通过标准“测试仪”线束进行测量
（以美元计算的平均绝对误差 + R²）并在最终排行榜上排名。

## 我们正在测量什么

**任务：**给定“item.summary”（产品的文本描述），预测其美元价格。

**指标：**“BENCHMARK_SIZE”测试项目的平均绝对误差（“$”）。越低越好。
第二个指标是 R²——接近 100% 意味着预测与真实价格相关。

**正在测试的提供商/模型：**

|供应商|型号|类型 |
|----------|------|------|
|开放路由器 | `openai/gpt-4.1-nano` |付费（小，快）|
|开放路由器 | `openai/gpt-4o-mini` |付费（平衡）|
|格罗克 | `llama-3.3-70b-多功能` |免费OSS（大）|
|格罗克 | `llama-3.1-8b-instant` |免费 OSS（小、快）|
|格罗克 | `deepseek-r1-distill-llama-70b` |免费OSS（推理）|

**使用的凭证：** OpenRouter 的`OR_API_KEY` + `OR_CLIENT_URL`； Groq 的“GROQ_API_KEY”+“GROQ_CLIENT_URL”。
这两个 API 都兼容 OpenAI，因此我们使用“openai.OpenAI(api_key=..., base_url=...)”。

In [ ]:
# 设置：导入、回购根解析、定价模块路径
import os
import re
import sys
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import r2_score
import pandas as pd


def _repo_root() -> Path:
    """Walk up from cwd until week6/pricer is found."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "week6" / "pricer").is_dir():
            return cand
    raise RuntimeError(
        "Cannot find week6/pricer — start Jupyter from the llm_engineering repo root."
    )


REPO_ROOT = _repo_root()

# 将 week6/ 添加到 sys.path，以便“from Pricer.items import Item”适用于任何 CWD
WEEK6_PATH = str(REPO_ROOT / "week6")
if WEEK6_PATH not in sys.path:
    sys.path.insert(0, WEEK6_PATH)

from pricer.items import Item
from pricer.evaluator import Tester

BENCHMARK_SIZE = 50   # items per model — raise to 200 for a more reliable run
WORKERS = 3

print(f"Repo root : {REPO_ROOT}")
print(f"Week6 path: {WEEK6_PATH}")
print(f"Benchmark : {BENCHMARK_SIZE} items per model, {WORKERS} workers")


In [ ]:
# .env / key check — 与 week4/day5.ipynb 和 week5/day5.ipynb 样式相同
load_dotenv(REPO_ROOT / ".env", override=True)

hf_token         = os.getenv("HF_TOKEN")
openai_api_key   = os.getenv("OPENAI_API_KEY")
or_api_key       = os.getenv("OR_API_KEY")
or_client_url    = os.getenv("OR_CLIENT_URL", "https://openrouter.ai/api/v1")
groq_api_key     = os.getenv("GROQ_API_KEY")
groq_client_url  = os.getenv("GROQ_CLIENT_URL", "https://api.groq.com/openai/v1")

for name, val in [
    ("HF_TOKEN",        hf_token),
    ("OPENAI_API_KEY",  openai_api_key),
    ("OR_API_KEY",      or_api_key),
    ("GROQ_API_KEY",    groq_api_key),
]:
    if val:
        print(f"{name} exists and begins {val[:8]}")
    else:
        print(f"{name} NOT SET — some models will be unavailable")


In [ ]:
# 两个 OpenAI 兼容客户端 — 每个提供商一个（与 week4/day5.ipynb 模式相同）

# OpenRouter：通往付费前沿模型的路线（GPT-4.1-nano、GPT-4o-mini，...）
or_client = OpenAI(
    api_key=or_api_key,
    base_url=or_client_url,
)

# Groq：免费 OSS 模型的快速推理（Llama、DeepSeek、Mixtral...）
groq_client = OpenAI(
    api_key=groq_api_key,
    base_url=groq_client_url,
)

print("OpenRouter client  :", or_client.base_url)
print("Groq client        :", groq_client.base_url)


In [ ]:
# 从 HuggingFace 加载精选的亚马逊产品数据集（第 1 天数据）
# 使用“lite”变体（~22k train / 1k val / 1k test）来保持低成本。
# 切换到“ed-donner/items_full”以获得更具代表性的基准。

from huggingface_hub import login

if hf_token:
    login(hf_token, add_to_git_credential=True)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN — cannot load dataset")

DATASET = "ed-donner/items_lite"

train, val, test = Item.from_hub(DATASET)
print(f"Loaded  {len(train):,} train | {len(val):,} val | {len(test):,} test items")
print(f"Sample  : {test[0]}")
print(f"Summary : {test[0].summary[:120]}...")


In [ ]:
# 快速构建 — 与 day4.ipynb 单元 20 相同
# 单一零样本用户消息：描述产品、询问价格。

def messages_for(item: Item) -> list[dict]:
    """Build the messages list for zero-shot price estimation."""
    return [
        {
            "role": "user",
            "content": (
                "Estimate the price of this product. "
                "Respond with the price only, no explanation.\n\n"
                + item.summary
            ),
        }
    ]


# 对第一个测试项目进行健全性检查
print(messages_for(test[0]))
print(f"\nActual price: ${test[0].price:.2f}")


In [ ]:
# Pricer 工厂 — 为任何 OpenAI 兼容客户端构建命名的定价函数。
# Tester 类从函数名称派生显示标题，因此我们使用
# 动态闭包，因此每个定价器都有一个唯一的 __name__。

def make_pricer(client: OpenAI, model: str, max_tokens: int = 10):
    """
    Return a callable `fn(item) -> str` that calls `client` with `model`.

    The function name is set to the model slug so Tester.make_title() renders it nicely.
    """
    def pricer(item: Item) -> str:
        response = client.chat.completions.create(
            model=model,
            messages=messages_for(item),
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content

    # 为函数指定一个测试程序标题的描述性名称
    pricer.__name__ = model.replace("/", "_").replace("-", "_").replace(".", "_")
    pricer.__qualname__ = pricer.__name__
    return pricer


# ── OpenRouter（付费）────────────────────────────────────────────────────────
gpt_4_1_nano_or = make_pricer(or_client,   "openai/gpt-4.1-nano")
gpt_4o_mini_or  = make_pricer(or_client,   "openai/gpt-4o-mini")

# ── Groq（免费OSS）──────────────────────────────────────────────────────────
llama_33_70b        = make_pricer(groq_client, "llama-3.3-70b-versatile")
llama_31_8b         = make_pricer(groq_client, "llama-3.1-8b-instant")
deepseek_r1_distill = make_pricer(groq_client, "deepseek-r1-distill-llama-70b")

# 注册表：显示名称→（函数、提供者）
PRICERS: dict[str, tuple] = {
    "GPT-4.1-nano (OpenRouter)":     (gpt_4_1_nano_or,    "OpenRouter"),
    "GPT-4o-mini (OpenRouter)":      (gpt_4o_mini_or,     "OpenRouter"),
    "Llama-3.3-70b (Groq)":          (llama_33_70b,       "Groq"),
    "Llama-3.1-8b (Groq)":           (llama_31_8b,        "Groq"),
    "DeepSeek-R1-Distill (Groq)":    (deepseek_r1_distill,"Groq"),
}

print(f"Registered {len(PRICERS)} pricers:")
for display_name, (fn, provider) in PRICERS.items():
    print(f"  [{provider}] {display_name}")


In [ ]:
# 运行基准测试——每个模型一次测试通过。
# 测试器同时运行 BENCHMARK_SIZE 项目（WORKERS 线程），打印
# 对每个项目的错误进行颜色编码，然后显示错误趋势+散点图。

benchmark_results: dict[str, dict] = {}

for display_name, (fn, provider) in PRICERS.items():
    print(f"\n{'='*60}")
    print(f"  {display_name}")
    print(f"{'='*60}")
    try:
        t = Tester(fn, test, title=display_name, size=BENCHMARK_SIZE, workers=WORKERS)
        t.run()
        avg_err = sum(t.errors) / len(t.errors)
        r2      = r2_score(t.truths, t.guesses) * 100
        benchmark_results[display_name] = {
            "provider": provider,
            "avg_error": avg_err,
            "r2": r2,
            "errors": t.errors,
            "guesses": t.guesses,
            "truths": t.truths,
        }
    except Exception as exc:
        print(f"  ERROR: {exc}")
        benchmark_results[display_name] = {
            "provider": provider,
            "avg_error": float("inf"),
            "r2": float("-inf"),
            "errors": [],
            "guesses": [],
            "truths": [],
        }

print(f"\nBenchmark complete. {len(benchmark_results)} models evaluated.")


In [ ]:
# 结果排行榜——按平均绝对误差排名（越低=越好）

rows = []
for display_name, r in benchmark_results.items():
    if r["avg_error"] == float("inf"):
        row = {
            "Model": display_name,
            "Provider": r["provider"],
            "Avg Error ($)": "ERROR",
            "R² (%)": "—",
        }
    else:
        row = {
            "Model": display_name,
            "Provider": r["provider"],
            "Avg Error ($)": f"{r['avg_error']:.2f}",
            "R² (%)": f"{r['r2']:.1f}",
        }
    rows.append(row)

# 排序：有错误的模型排在最后，其他模型按 avg_error 升序排序
def sort_key(row):
    try:
        return float(row["Avg Error ($)"])
    except (ValueError, TypeError):
        return float("inf")

rows.sort(key=sort_key)

print(f"\n{'='*70}")
print(f"{'LEADERBOARD':^70}")
print(f"{'='*70}")
print(f"{'Rank':<5} {'Model':<35} {'Provider':<12} {'Avg Err ($)':>12} {'R²':>8}")
print("-" * 70)

for rank, row in enumerate(rows, 1):
    err = row["Avg Error ($)"]
    r2  = row["R² (%)"]
    print(f"{rank:<5} {row['Model']:<35} {row['Provider']:<12} {err:>12} {r2:>8}")

print("-" * 70)

# 最佳模型总结
if rows and rows[0]["Avg Error ($)"] != "ERROR":
    winner = rows[0]
    print(f"\nBest model : {winner['Model']}")
    print(f"Provider   : {winner['Provider']}")
    print(f"Avg Error  : ${winner['Avg Error ($)']}")
    print(f"R²         : {winner['R² (%)']}")


## 主要发现

- **OpenRouter** 将呼叫路由到官方 OpenAI 端点；对于结构化数值任务，GPT-4o-mini 通常可以很好地平衡成本和准确性。
- **Groq** 为开放权重模型提供极快的推理； Llama-3.3-70b 通常是用于价格估算的最强 OSS 选项。
- **DeepSeek-R1-Distill**（推理模型）可能会过度考虑简单的回归任务，有时会产生详细的输出 - 测试器会删除它找到的第一个数字，但要注意解析失败。
- 第 5 天微调的“gpt-4.1-nano”变体（在同一数据集中的 100 个示例上进行训练）始终优于零样本模型，说明了即使使用最少的数据，特定于任务的微调的价值。

### 后续步骤（第 5 天方向）
要微调此数据，请将训练项目转换为 JSONL 并上传到 OpenAI（或您选择的提供商）：

In [ ]:
# 第 5 天预览：用于微调的 JSONL 格式（无需上传 - 仅显示格式）

def make_jsonl(items: list, n: int = 5) -> str:
    """Produce JSONL fine-tuning data in OpenAI chat format."""
    lines = []
    for item in items[:n]:
        messages = [
            {"role": "user", "content": (
                "Estimate the price of this product. "
                "Respond with the price only, no explanation.\n\n"
                + item.summary
            )},
            {"role": "assistant", "content": f"${item.price:.2f}"},
        ]
        lines.append(json.dumps({"messages": messages}))
    return "\n".join(lines)


sample_jsonl = make_jsonl(train, n=5)
print("=== Sample fine-tuning JSONL (first 5 items) ===\n")
print(sample_jsonl)
print(f"\n... and {len(train):,} more training examples available.")
print("\nTo fine-tune, upload via:")
print('  openai.files.create(open("fine_tune_train.jsonl","rb"), purpose="fine-tune")')
print('  openai.fine_tuning.jobs.create(training_file=..., model="gpt-4.1-nano-2025-04-14")')
